# Construeix el dataset analític aplanat

## 0. Imports necessaris

In [56]:
import os
import pandas as pd

## 1. Carregar les taules

In [57]:
print("Cargando cohort.xlsx ...")
cohort = pd.read_excel("raw/cohort.xlsx")
print(f"  cohort: {cohort.shape[0]} filas, {cohort.shape[1]} columnas")

print("Cargando diagnostics.xlsx ...")
diagnostics = pd.read_excel("raw/diagnostics.xlsx")
print(f"  diagnostics: {diagnostics.shape[0]} filas, {diagnostics.shape[1]} columnas")

print("Cargando farmacs.xlsx ...")
farmacs = pd.read_excel("raw/farmacs.xlsx")
print(f"  farmacs: {farmacs.shape[0]} filas, {farmacs.shape[1]} columnas")

Cargando cohort.xlsx ...
  cohort: 37902 filas, 5 columnas
Cargando diagnostics.xlsx ...
  diagnostics: 37902 filas, 10 columnas
Cargando farmacs.xlsx ...
  farmacs: 23580 filas, 16 columnas


### 1.1 Eliminem informació irrellevant per l'estudi

In [58]:
# Eliminem la situació del pacient (mort o viu)
cohort = cohort.drop(columns="situacio")

### 1.2 Afegim la columna "target" on 0 = No Cronic, 1 = PCC, 2 = MACA

In [59]:
cohort["target"] = cohort["cronic"].map({"NO":0, "PCC":1, "MACA":2})

## 2. Ajuntem les taules
### 2.2 Left Join 1:1

In [60]:
print("\nRealizando LEFT JOIN cohort ← diagnostics (on id_pacient) ...")
df = cohort.merge(diagnostics, on="id_pacient", how="left")
print(f"  Resultado parcial: {df.shape[0]} filas, {df.shape[1]} columnas")

# Emplenem els valors NaN amb zeros. Assumim que si és NaN, no tenen farmacs
farmacs = farmacs.fillna(0)

print("Realizando LEFT JOIN resultado ← farmacs (on id_pacient) ...")
df = df.merge(farmacs, on="id_pacient", how="left")
print(f"  Resultado parcial: {df.shape[0]} filas, {df.shape[1]} columnas")


Realizando LEFT JOIN cohort ← diagnostics (on id_pacient) ...
  Resultado parcial: 37902 filas, 14 columnas
Realizando LEFT JOIN resultado ← farmacs (on id_pacient) ...
  Resultado parcial: 37902 filas, 29 columnas


## 3. Agregacions 1:N — recompte de visites per pacient
### 3.1 Visites Hospital (Intervals de dies)

In [61]:
print("\nCargant raw/visites_hospital.xlsx ...")
hosp = pd.read_excel("raw/visites_hospital.xlsx")
print(f"  visites_hospital: {hosp.shape[0]} files")

# Comptem les visites per a cada interval de dies 1-121, 122-242, 243-365
hosp_1_121 = hosp[hosp["data"].between(1, 121)].groupby("id_pacient").size().rename("visites_hosp_1_121")
hosp_122_242 = hosp[hosp["data"].between(122, 242)].groupby("id_pacient").size().rename("visites_hosp_122_242")
hosp_243_365 = hosp[hosp["data"].between(243, 365)].groupby("id_pacient").size().rename("visites_hosp_243_365")

# Ajuntem les 3 columnes noves al dataset principal
for col, series in [("visites_hosp_1_121", hosp_1_121), 
                    ("visites_hosp_122_242", hosp_122_242), 
                    ("visites_hosp_243_365", hosp_243_365)]:
    df = df.merge(series, on="id_pacient", how="left")

    #Si un pacient no tenia cap visita posem el valor a 0
    df[col] = df[col].fillna(0).astype(int)
    print(f"  Columna '{col}' afegida.")



Cargant raw/visites_hospital.xlsx ...
  visites_hospital: 10163 files
  Columna 'visites_hosp_1_121' afegida.
  Columna 'visites_hosp_122_242' afegida.
  Columna 'visites_hosp_243_365' afegida.


In [62]:
df

,id_pacient,sexe,cronic,grup_edat,target,altres,problemes_salut_aguts,problemes_salut_anomalia_congenita,problemes_salut_cronics,problemes_salut_indefinits,...,sistema_digestiu_i_metabolisme,sistema_genitourinari_i_hormones_sexuals,sistema_musculoesqueletic,sistema_nervios,sistema_respiratori,organs_dels_sentits,farmacs_totals,visites_hosp_1_121,visites_hosp_122_242,visites_hosp_243_365
0,1,D,NO,80-85,0,4,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2,H,NO,75-80,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,3,D,NO,70-75,0,4,0,0,0,0,...,0.0,0.0,3.0,1.0,0.0,0.0,4.0,0,0,0
3,4,H,NO,80-85,0,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,5,D,NO,70-75,0,4,0,0,0,0,...,1.0,1.0,2.0,1.0,0.0,0.0,8.0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37897,37898,D,NO,75-80,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
37898,37899,D,NO,80-85,0,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
37899,37900,D,NO,70-75,0,9,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0
37900,37901,H,PCC,80-85,1,11,0,0,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0,1,0


### 3.2 Visites Atenció Intermèdia (Intervals de dies)

In [63]:
print("\nCargando raw/visites_intermedia.xlsx ...")
inter = pd.read_excel("raw/visites_intermedia.xlsx")
print(f"  visites_intermedia: {inter.shape[0]} files")

# Comptem les visites per a cada interval de dies
inter_1_121 = inter[inter["data"].between(1, 121)].groupby("id_pacient").size().rename("visites_inter_1_121")
inter_122_242 = inter[inter["data"].between(122, 242)].groupby("id_pacient").size().rename("visites_inter_122_242")
inter_243_365 = inter[inter["data"].between(243, 365)].groupby("id_pacient").size().rename("visites_inter_243_365")

# Ajuntem les 3 columnes noves al dataset principal
for col, series in [("visites_inter_1_121", inter_1_121), 
                    ("visites_inter_122_242", inter_122_242), 
                    ("visites_inter_243_365", inter_243_365)]:
    df = df.merge(series, on="id_pacient", how="left")

    #Si un pacient no tenia cap visita posem el valor a 0
    df[col] = df[col].fillna(0).astype(int)
    print(f"  Columna '{col}' afegida.")


Cargando raw/visites_intermedia.xlsx ...
  visites_intermedia: 2465 files
  Columna 'visites_inter_1_121' afegida.
  Columna 'visites_inter_122_242' afegida.
  Columna 'visites_inter_243_365' afegida.


In [64]:
df

,id_pacient,sexe,cronic,grup_edat,target,altres,problemes_salut_aguts,problemes_salut_anomalia_congenita,problemes_salut_cronics,problemes_salut_indefinits,...,sistema_nervios,sistema_respiratori,organs_dels_sentits,farmacs_totals,visites_hosp_1_121,visites_hosp_122_242,visites_hosp_243_365,visites_inter_1_121,visites_inter_122_242,visites_inter_243_365
0,1,D,NO,80-85,0,4,0,0,0,0,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
1,2,H,NO,75-80,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
2,3,D,NO,70-75,0,4,0,0,0,0,...,1.0,0.0,0.0,4.0,0,0,0,0,0,0
3,4,H,NO,80-85,0,1,0,0,0,0,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
4,5,D,NO,70-75,0,4,0,0,0,0,...,1.0,0.0,0.0,8.0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37897,37898,D,NO,75-80,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
37898,37899,D,NO,80-85,0,1,0,0,0,0,...,NaN,NaN,NaN,NaN,0,0,0,0,0,0
37899,37900,D,NO,70-75,0,9,0,0,1,0,...,0.0,0.0,0.0,1.0,0,0,0,0,0,0
37900,37901,H,PCC,80-85,1,11,0,0,0,0,...,1.0,0.0,0.0,1.0,0,1,0,0,0,0


### 3.3 Visites Urgències (Risc Vital Potencial)

In [65]:
print("\nCargando raw/visites_urgencies.xlsx ...")
urg = pd.read_excel("raw/visites_urgencies.xlsx")
print(f"  visites_urgencies: {urg.shape[0]} files")

# Filtrem només les files amb el nivell de triatge i comptem per pacient
urg_risc = urg[urg["nivell_triatge"] == "Risc vital potencial"].groupby("id_pacient").size().rename("visites_urgencies_risc_vital")

df = df.merge(urg_risc, on="id_pacient", how="left")
df["visites_urgencies_risc_vital"] = df["visites_urgencies_risc_vital"].fillna(0).astype(int)
print("  Columna 'visites_urgencies_risc_vital' afegida.")


Cargando raw/visites_urgencies.xlsx ...
  visites_urgencies: 19529 files
  Columna 'visites_urgencies_risc_vital' afegida.


### 3.4 Visites Atenció Primària (Total de visites)

In [66]:
print("\nCargando raw/visites_primaria.xlsx ...")
primaria = pd.read_excel("raw/visites_primaria.xlsx")
print(f"  visites_primaria: {primaria.shape[0]} filas")

# Sumem la columna "visites" per pacient
prim_visites = primaria.groupby("id_pacient")["visites"].sum().rename("num_visitas_primaria")

df = df.merge(prim_visites, on="id_pacient", how="left")
df["num_visitas_primaria"] = df["num_visitas_primaria"].fillna(0).astype(int)
print("  Columna 'num_visitas_primaria' añadida.")


Cargando raw/visites_primaria.xlsx ...
  visites_primaria: 98633 filas
  Columna 'num_visitas_primaria' añadida.


## 4. Variables de laboratori — mean i slope per prova

In [67]:
print("\nCargando laboratori.xlsx ...")
lab = pd.read_excel("raw/laboratori.xlsx")
print(f"  laboratori: {lab.shape[0]} files, {lab['id_pacient'].nunique()} pacients, {lab['desc_prova_ics'].nunique()} proves")


# Nom curt per a cada prova (per utilitzar com a sufix de columna)
SHORT_NAMES = {
    "GLUCOSA-SÈRUM":                                  "glucosa",
    "UREA-SÈRUM":                                     "urea",
    "PROTEÏNA C REACTIVA (PCR)-SÈRUM":                "pcr",
    "BILIRUBINA-SÈRUM":                               "bilirubina",
    "ASPARTAT AMINOTRANSFERASA-SÈRUM":                "ast",
    "ALANINA AMINOTRANSFERASA-SÈRUM":                 "alt",
    "ALBÚMINA-SÈRUM":                                 "albumina",
    "COLESTEROL-SÈRUM":                               "colesterol",
    "FOSFATASA ALCALINA-SÈRUM":                       "fosfatasa",
    "PROTEÏNA-SÈRUM":                                 "proteina",
    "TIROTROPINA-SÈRUM":                              "tsh",
    "ERITROSEDIMENTACIÓ (VSG)-SANG":                  "vsg",
    "ÀCID FÒLIC-SÈRUM":                               "ac_folico",
    "COBALAMINES (VITAMINA B12)-SÈRUM":               "vit_b12",
    "FERRITINA-SÈRUM":                                "ferritina",
    "FERRO-SÈRUM":                                    "ferro",
    "PRO-BNP-SÈRUM":                                  "pro_bnp",
    "DÍMER D DE LA FIBRINA (IMMUNOTURBIDIMETRIA)-PLASMA": "dimero_d",
}

lab["prova_short"] = lab["desc_prova_ics"].map(SHORT_NAMES)


Cargando laboratori.xlsx ...
  laboratori: 15282 files, 3312 pacients, 18 proves


In [68]:
# Pivotar: una columna per prova × mean
lab_mean = lab.pivot_table(index="id_pacient", columns="prova_short", values="mean").add_suffix("_mean")

# CREEM ELS INDICADORS: Indiquem si el valor de la mitjana és NaN o no, és a dir si s'ha fet la prova o no, convertim True/False a 1/0
lab_done = lab_mean.notna().astype(int)
lab_done.columns = [f"{c}_realitzada" for c in lab_done.columns]

# Emplenem els valors nulls amb la moda de cada columna i afegim l'indicador
# lab_mean = lab_mean.fillna(lab_mean.mode().iloc[0])
lab_mean = lab_mean.join(lab_done)
print (lab_mean.columns)

Index(['ac_folico_mean', 'albumina_mean', 'alt_mean', 'ast_mean',
       'bilirubina_mean', 'colesterol_mean', 'dimero_d_mean', 'ferritina_mean',
       'ferro_mean', 'fosfatasa_mean', 'glucosa_mean', 'pcr_mean',
       'pro_bnp_mean', 'proteina_mean', 'tsh_mean', 'urea_mean',
       'vit_b12_mean', 'vsg_mean', 'ac_folico_mean_realitzada',
       'albumina_mean_realitzada', 'alt_mean_realitzada',
       'ast_mean_realitzada', 'bilirubina_mean_realitzada',
       'colesterol_mean_realitzada', 'dimero_d_mean_realitzada',
       'ferritina_mean_realitzada', 'ferro_mean_realitzada',
       'fosfatasa_mean_realitzada', 'glucosa_mean_realitzada',
       'pcr_mean_realitzada', 'pro_bnp_mean_realitzada',
       'proteina_mean_realitzada', 'tsh_mean_realitzada',
       'urea_mean_realitzada', 'vit_b12_mean_realitzada',
       'vsg_mean_realitzada'],
      dtype='str')


In [69]:
# Pivotar: una columna per prova × slope
lab_slope = lab.pivot_table(index="id_pacient", columns="prova_short", values="slope").add_suffix("_slope")

# CREEM ELS INDICADORS: Indiquem si el valor de la mitjana és NaN o no, és a dir si s'ha fet la prova o no, convertim True/False a 1/0
slope_done = lab_slope.notna().astype(int)
slope_done.columns = [f"{c}_registrat" for c in slope_done.columns]

# Emplenem els valors nulls amb la moda de cada columna i afegim l'indicador
# lab_slope = lab_slope.fillna(lab_slope.mode().iloc[0])
lab_slope = lab_slope.join(slope_done)
print (lab_slope.columns)

Index(['ac_folico_slope', 'albumina_slope', 'alt_slope', 'ast_slope',
       'bilirubina_slope', 'colesterol_slope', 'ferritina_slope',
       'ferro_slope', 'fosfatasa_slope', 'glucosa_slope', 'pcr_slope',
       'pro_bnp_slope', 'proteina_slope', 'tsh_slope', 'urea_slope',
       'vit_b12_slope', 'vsg_slope', 'ac_folico_slope_registrat',
       'albumina_slope_registrat', 'alt_slope_registrat',
       'ast_slope_registrat', 'bilirubina_slope_registrat',
       'colesterol_slope_registrat', 'ferritina_slope_registrat',
       'ferro_slope_registrat', 'fosfatasa_slope_registrat',
       'glucosa_slope_registrat', 'pcr_slope_registrat',
       'pro_bnp_slope_registrat', 'proteina_slope_registrat',
       'tsh_slope_registrat', 'urea_slope_registrat',
       'vit_b12_slope_registrat', 'vsg_slope_registrat'],
      dtype='str')


In [70]:
# Ajuntem tot en una sola taula
lab_pivot = lab_mean.join(lab_slope).reset_index()
print(f"  Columnas de laboratorio generadas: {lab_pivot.shape[1] - 1}")

# Left join
df = df.merge(lab_pivot, on="id_pacient", how="left")
print(f"  Dataset tras añadir laboratorio: {df.shape[0]} filas, {df.shape[1]} columnas")

  Columnas de laboratorio generadas: 70
  Dataset tras añadir laboratorio: 37902 filas, 107 columnas


In [71]:
# 1. Posa a 0 els indicadors de realitzat/registrat que siguin nuls (pacients sense analítiques)
done_cols = [c for c in df.columns if c.endswith("_realitzada") or c.endswith("_registrat")]
df[done_cols] = df[done_cols].fillna(0).astype(int)

# # 2. Emplena els valors de mean i slope amb la moda de la columna
# val_cols = [c for c in df.columns if c.endswith("_mean") or c.endswith("_slope")]
# for col in val_cols:
#     mode_vals = df[col].mode()
#     if not mode_vals.empty:
#         df[col] = df[col].fillna(mode_vals[0])


## 5. Resum

In [72]:
print(f"\n{'='*50}")
print(f"=== Dataset resultant ===")
print(f"  Files:    {df.shape[0]}")
print(f"  Columnes: {df.shape[1]}")
print(f"  Noms de les columnes: {list(df.columns)}")

print(f"\n  Estadístiques de visites:")
visit_cols = [
    "visites_hosp_1_121", "visites_hosp_122_242", "visites_hosp_243_365",
    "visites_inter_1_121", "visites_inter_122_242", "visites_inter_243_365",
    "visites_urgencies_risc_vital", "num_visitas_primaria"
]
for col in visit_cols:
    if col in df.columns:
        print(f"    {col}: mitjana={df[col].mean():.2f}, màx={df[col].max()}")

# Identifiquem les columnes de laboratori (incloses les de control/realització si cal)
lab_cols = [c for c in df.columns if any(c.endswith(s) for s in ["_mean", "_slope", "_realitzada", "_registrat"])]

print(f"\n  Cobertura de laboratori (pacients amb dades):")
for col in sorted(lab_cols):
    non_null = df[col].notna().sum()
    print(f"    {col}: {non_null} ({100*non_null/len(df):.1f}%)")

print(f"\n  Valors nuls per columna (no-lab):")
nulls = df.isnull().sum()
for col in df.columns:
    if nulls[col] > 0 and col not in lab_cols:
        print(f"    {col}: {nulls[col]} nuls ({100*nulls[col]/len(df):.1f}%)")



=== Dataset resultant ===
  Files:    37902
  Columnes: 107
  Noms de les columnes: ['id_pacient', 'sexe', 'cronic', 'grup_edat', 'target', 'altres', 'problemes_salut_aguts', 'problemes_salut_anomalia_congenita', 'problemes_salut_cronics', 'problemes_salut_indefinits', 'problemes_salut_neoplasia_benigna', 'problemes_salut_neoplasia_maligna', 'signes_i_sintomes', 'diags_totals', 'antiinfecciosos_per_a_us_sistemic', 'antineoplasics_i_immunomoduladors', 'dermatologics', 'diversos', 'preparats_hormonals_sistemics', 'productes_antiparasitaris,_insecticides_i_repellents', 'sang_i_organs_hematopoetics', 'sistema_cardiovascular', 'sistema_digestiu_i_metabolisme', 'sistema_genitourinari_i_hormones_sexuals', 'sistema_musculoesqueletic', 'sistema_nervios', 'sistema_respiratori', 'organs_dels_sentits', 'farmacs_totals', 'visites_hosp_1_121', 'visites_hosp_122_242', 'visites_hosp_243_365', 'visites_inter_1_121', 'visites_inter_122_242', 'visites_inter_243_365', 'visites_urgencies_risc_vital', 'num

    proteina_slope: 157 (0.4%)
    proteina_slope_registrat: 37902 (100.0%)
    tsh_mean: 133 (0.4%)
    tsh_mean_realitzada: 37902 (100.0%)
    tsh_slope: 29 (0.1%)
    tsh_slope_registrat: 37902 (100.0%)
    urea_mean: 3276 (8.6%)
    urea_mean_realitzada: 37902 (100.0%)
    urea_slope: 1730 (4.6%)
    urea_slope_registrat: 37902 (100.0%)
    vit_b12_mean: 96 (0.3%)
    vit_b12_mean_realitzada: 37902 (100.0%)
    vit_b12_slope: 10 (0.0%)
    vit_b12_slope_registrat: 37902 (100.0%)
    vsg_mean: 128 (0.3%)
    vsg_mean_realitzada: 37902 (100.0%)
    vsg_slope: 46 (0.1%)
    vsg_slope_registrat: 37902 (100.0%)

  Valors nuls per columna (no-lab):
    antiinfecciosos_per_a_us_sistemic: 14322 nuls (37.8%)
    antineoplasics_i_immunomoduladors: 14322 nuls (37.8%)
    dermatologics: 14322 nuls (37.8%)
    diversos: 14322 nuls (37.8%)
    preparats_hormonals_sistemics: 14322 nuls (37.8%)
    productes_antiparasitaris,_insecticides_i_repellents: 14322 nuls (37.8%)
    sang_i_organs_hematopoe

## 6. Desar

In [73]:
# 1. Desar en format Excel (per a anàlisi visual)
output_path = "processed/dataset_analitico.xlsx"
print(f"\nGuardando en {output_path} ...")
df.to_excel(output_path, index=False)
print(f"¡Guardado correctamente! ({output_path})")


# 2. Desar en format CSV (per a entrenar els models d'IA)
output_path_csv = "processed/dataset_final_pcc.csv"
print(f"Guardando en {output_path_csv} ...")
df.to_csv(output_path_csv, index=False)
print(f"¡Guardado correctamente! ({output_path_csv})")


Guardando en processed/dataset_analitico.xlsx ...
¡Guardado correctamente! (processed/dataset_analitico.xlsx)
Guardando en processed/dataset_final_pcc.csv ...
¡Guardado correctamente! (processed/dataset_final_pcc.csv)
